<a href="https://colab.research.google.com/github/MariamWassim/internship-/blob/main/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MariamWassim/internship-/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

**Lane (locked): CTR / Engagement Opportunity Scoring.** Same table and grain as `w03_data_contract.ipynb` — one row per `(client_hash_id, content_hash_id)` for March 2026, from `fact_content_daily_performance`.

> Skills loaded for this notebook: `building-baselines/SKILL.md` + `flyrank/flyrank-data/SKILL.md`.


In [ ]:
# Rebuild the page-month frame from w03 — same contract, same month, same exclusions.
%pip -q install duckdb

import os
import duckdb
import numpy as np
import pandas as pd

HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
FACT_MAR = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')"

page_month = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS impressions_mar,
        SUM(gsc_clicks) AS clicks_mar,
        AVG(CASE WHEN gsc_avg_position > 0 THEN gsc_avg_position END) AS avg_position_mar
    FROM {FACT_MAR}
    GROUP BY 1, 2
    HAVING impressions_mar >= 100
       AND avg_position_mar IS NOT NULL
""").df()

page_month['ctr_mar'] = page_month['clicks_mar'] / page_month['impressions_mar']

def position_tier(p):
    if p <= 3: return '1-3'
    if p <= 10: return '4-10'
    if p <= 20: return '11-20'
    return '21+'

page_month['position_tier'] = page_month['avg_position_mar'].apply(position_tier)
print(f'{len(page_month):,} pages loaded — same base frame as w03.')


## 1. My rule and its reason codes — plus the two signal checks it leans on

**The rule, in plain words:** *a page is worth a CTR-fix review if it's under-clicking for its position tier by a real margin, AND it has enough impression volume that the gap isn't just noise.*

That rule leans on two claims. Before coding it, check both — a clearly-explained negative here is a win, not a failure.

### Signal check A — CTR really does track position tier (flag-linked: this is the exact belief behind FlyRank's `low_ctr_visible_page` flag — impressions ≥ 500, position 1–20, CTR < 0.5)

**Claim:** better position tier → higher CTR. If this is false or muddy, a position-tier-relative CTR gap is the wrong yardstick entirely.

**Test:** CTR **weighted by impressions** per tier (never the mean of per-page rates — a page with 2 clicks / 4 impressions and a page with 2,000 clicks / 4,000 impressions are not equally informative; per the signal-audit skill, averaging per-row rates is not the true rate).


In [ ]:
signal_a = page_month.groupby('position_tier').agg(
    n=('content_hash_id', 'size'),
    total_impressions=('impressions_mar', 'sum'),
    total_clicks=('clicks_mar', 'sum'),
).reset_index()
signal_a['weighted_ctr'] = signal_a['total_clicks'] / signal_a['total_impressions']

tier_order = ['1-3', '4-10', '11-20', '21+']
signal_a['position_tier'] = pd.Categorical(signal_a['position_tier'], categories=tier_order, ordered=True)
signal_a = signal_a.sort_values('position_tier')

print('Signal A — weighted CTR by position tier (n = pages per tier):')
print(signal_a[['position_tier', 'n', 'weighted_ctr']].to_string(index=False))

# Sample-size floor per the signal-audit skill: no verdict from a bucket under ~50 rows.
below_floor = signal_a[signal_a['n'] < 50]
if len(below_floor):
    print('\n⚠ Tier(s) below the n=50 floor — do not verdict on these alone:', below_floor['position_tier'].tolist())


**Verdict A: `[FILL IN AFTER RUNNING]`**

How to read the table above and choose honestly:
- **CONFIRMED** — `weighted_ctr` decreases monotonically (or nearly so) from `1-3` down to `21+`, and every tier clears the n=50 floor.
- **OPPOSITE** — CTR is flat or *increases* in worse tiers (can happen with a branded/navigational query mix skewing a tier — worth a one-line note if so).
- **MIXED** — the direction holds for some adjacent tiers but not all (e.g. `1-3` > `4-10`, but `11-20` ≈ `21+`).
- **FALSE** — no meaningful separation between tiers at all.

Whatever the real table above says, write one sentence here: what it means for using position-tier as the yardstick for the rule below.

### Signal check B — the CTR-gap signal gets less noisy at higher volume (flag-linked: this is the logic behind `quick_win` / minimum-volume gating on every FlyRank flag)

**Claim:** a CTR gap computed from a page with 110 impressions is far less trustworthy than the same gap from a page with 5,000 — check whether the *spread* of the gap actually shrinks as volume rises.


In [ ]:
# Reuse the leave-one-out expected-CTR-per-tier from w03 to compute ctr_gap here too.
tier_mean = page_month.groupby('position_tier')['ctr_mar'].transform('mean')
tier_n = page_month.groupby('position_tier')['ctr_mar'].transform('count')
page_month['expected_ctr_tier'] = (tier_mean * tier_n - page_month['ctr_mar']) / (tier_n - 1)
page_month['ctr_gap'] = page_month['expected_ctr_tier'] - page_month['ctr_mar']

def impression_tier(n):
    if n < 500: return '100-499'
    if n < 3000: return '500-2999'
    return '3000+'

page_month['impression_tier'] = page_month['impressions_mar'].apply(impression_tier)

signal_b = page_month.groupby('impression_tier').agg(
    n=('content_hash_id', 'size'),
    mean_ctr_gap=('ctr_gap', 'mean'),
    std_ctr_gap=('ctr_gap', 'std'),
).reset_index()
imp_order = ['100-499', '500-2999', '3000+']
signal_b['impression_tier'] = pd.Categorical(signal_b['impression_tier'], categories=imp_order, ordered=True)
signal_b = signal_b.sort_values('impression_tier')

print('Signal B — CTR-gap spread by impression tier (n = pages per tier):')
print(signal_b.to_string(index=False))

below_floor_b = signal_b[signal_b['n'] < 50]
if len(below_floor_b):
    print('\n⚠ Tier(s) below the n=50 floor — do not verdict on these alone:', below_floor_b['impression_tier'].tolist())


**Verdict B: `[FILL IN AFTER RUNNING]`**

- **CONFIRMED** — `std_ctr_gap` drops clearly from `100-499` to `3000+`: low-volume gaps really are noisier, so the volume gate in the rule below is doing real work, not just being cautious for no reason.
- **OPPOSITE** — spread is flat or *higher* at high volume (would be surprising — worth double-checking the tier boundaries or a data quirk before trusting the rule's volume gate).
- **MIXED** — spread drops from low to mid volume but doesn't keep dropping into `3000+`.
- **FALSE** — no relationship between volume and spread.

If this comes back CONFIRMED, it directly justifies the `impressions_mar >= 500` gate in the rule below — that's not an arbitrary number, it's the volume tier the check above shows is meaningfully less noisy.


## 2. Build the ranked queue (writes the CSV)

The rule, coded exactly as stated above — one score, one reason code, one action label. No fitted weights.


In [ ]:
# Score: rewards a bigger CTR gap, dampened by log-volume so a handful of huge-volume pages
# don't just steamroll the ranking — readable, not fitted.
page_month['score'] = (page_month['ctr_gap'].clip(lower=0) * np.log1p(page_month['impressions_mar'])).round(3)

# ONE reason code per row — reuses FlyRank's own real flag name for continuity with the session.
has_gap = page_month['ctr_gap'] > 0
enough_volume = page_month['impressions_mar'] >= 500
good_position = page_month['avg_position_mar'] <= 20

page_month['reason_code'] = np.where(
    has_gap & enough_volume & good_position,
    'low_ctr_visible_page',
    'no_flag',
)

# ONE action label — binary, transparent, matches the reason code 1:1.
page_month['action'] = np.where(page_month['reason_code'] == 'low_ctr_visible_page', 'review_ctr_fix', 'monitor')

queue = page_month.sort_values('score', ascending=False).reset_index(drop=True)
queue.insert(0, 'rank', queue.index + 1)

print('Action mix:')
print(queue['action'].value_counts())
print(f"\nBase rate (share flagged review_ctr_fix): {(queue['action'] == 'review_ctr_fix').mean():.1%}")

out_cols = ['rank', 'client_hash_id', 'content_hash_id', 'impressions_mar', 'clicks_mar',
            'avg_position_mar', 'position_tier', 'ctr_mar', 'expected_ctr_tier', 'ctr_gap',
            'score', 'reason_code', 'action']

import os
os.makedirs('work/outputs', exist_ok=True)
queue[out_cols].to_csv('work/outputs/baseline_action_score.csv', index=False)
print(f"\nWrote {len(queue):,} rows to work/outputs/baseline_action_score.csv")
queue[out_cols].head(10)


## 3. Top-10 review

For each of the top 10: the action, why it's there, and what would make it wrong. Use the printed diagnostics below each row to write these honestly rather than restating the score.


In [ ]:
top10 = queue.head(10)
for _, r in top10.iterrows():
    print(f"#{r['rank']:>2} | action={r['action']:<15} reason={r['reason_code']}")
    print(f"     client={r['client_hash_id']}  content={r['content_hash_id']}")
    print(f"     impressions={r['impressions_mar']:.0f}  clicks={r['clicks_mar']:.0f}  "
          f"avg_position={r['avg_position_mar']:.1f}  tier={r['position_tier']}")
    print(f"     ctr={r['ctr_mar']:.3%}  expected_for_tier={r['expected_ctr_tier']:.3%}  "
          f"gap={r['ctr_gap']:.3%}  score={r['score']:.2f}")
    print()


Fill one line each, using the diagnostics printed above — real numbers, not the score restated:

1. **#1** — Action: `[review_ctr_fix / monitor]`. Why it's there: `[e.g. "X impressions at position Y, clicking at Z% vs an expected W% for that tier"]`. What would make it wrong: `[e.g. "if this page's title/meta already changed recently and the fix is already underway"]`.
2. **#2** — Action: . Why it's there: . What would make it wrong: .
3. **#3** — Action: . Why it's there: . What would make it wrong: .
4. **#4** — Action: . Why it's there: . What would make it wrong: .
5. **#5** — Action: . Why it's there: . What would make it wrong: .
6. **#6** — Action: . Why it's there: . What would make it wrong: .
7. **#7** — Action: . Why it's there: . What would make it wrong: .
8. **#8** — Action: . Why it's there: . What would make it wrong: .
9. **#9** — Action: . Why it's there: . What would make it wrong: .
10. **#10** — Action: . Why it's there: . What would make it wrong: .

Common honest "what would make it wrong" reasons worth considering per row: the client has very thin March history (check against `dim_clients.gsc_data_start`); the position average is being pulled by one anomalous day rather than a steady month; the page is genuinely branded/navigational, where a lower CTR than an informational-intent peer in the same position tier is expected, not a real opportunity.


## 4. Weak picks + leakage check


In [ ]:
# Weak-pick flags: mechanical checks a human should still read, not a replacement for judgment above.
top10_flagged = top10.copy()
top10_flagged['near_volume_floor'] = top10_flagged['impressions_mar'] < 700   # barely clears the 500 gate
top10_flagged['near_position_edge'] = top10_flagged['avg_position_mar'] > 18  # barely inside the <=20 cutoff

weak = top10_flagged[top10_flagged['near_volume_floor'] | top10_flagged['near_position_edge']]
print(f'{len(weak)} of the top 10 sit near a threshold edge — read these first, they are the most fragile picks:')
print(weak[['rank', 'impressions_mar', 'avg_position_mar', 'near_volume_floor', 'near_position_edge']].to_string(index=False))
if len(weak) == 0:
    print('None near an edge in the top 10 — look at ranks 11-20 by hand before trusting the queue is edge-free throughout.')


In [ ]:
# Leakage check — different question than w03's trap. This is a hand-written baseline, not a
# trained classifier: using ctr_mar / clicks_mar directly as scoring inputs is NOT leakage here,
# it's the whole point of a transparent rule. What WOULD be leakage for a baseline:

banned_substrings = ['apr', 'may', 'jun', 'future', 'next_', 'health_score', 'priority_score', 'action_type']
used_columns = list(page_month.columns)
leaked = [c for c in used_columns if any(b in c.lower() for b in banned_substrings)]

print('Columns used:', used_columns)
print('\nAny future-window or FlyRank product-flag columns present?', 'YES — FIX THIS' if leaked else 'None found.')
assert not leaked, f'Leakage risk: {leaked}'

print('\nConfirmed: every column in the score/reason-code/action logic comes from March 2026 only,')
print('and none of it is a FlyRank product decision flag (those are not shipped in this dataset anyway).')


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
